# Notebook 01: benchmark bảy họ và giao thức chia theo thời gian

Bài hỏi một benchmark phát hiện URL lừa đảo thực sự đo cái gì. Notebook này làm lại phần lõi: bảy họ mô hình trên bộ PhishVN (53.116 URL), chia ngẫu
nhiên và chia theo thời gian, các đối chứng của giao thức thời gian, cấu hình huấn luyện, dòng trùng lặp và tầng nhãn. Ba notebook sau làm phần còn lại:

| Notebook | Nội dung | Đọc kết quả của |
|---|---|---|
| `01_benchmark_and_protocol.ipynb` | benchmark bảy họ, chia theo thời gian và các đối chứng, dòng trùng, tầng nhãn | |
| `02_transfer.ipynb` | ma trận chuyển giao giữa bốn corpus, huấn luyện gộp, khoảng tin cậy, SHAP | 01 |
| `03_robustness.ipynb` | tối ưu siêu tham số, stacking, dự báo độ trôi, nhãn nhiễu | 01 |
| `04_cnn_and_dedup.ipynb` | CNN trên ký tự, bỏ dòng trùng, bỏ tiền tố `www.`, tổng hợp cả bốn notebook | 01, 02 |

Chạy theo thứ tự 01 → 04, mỗi notebook một lần chạy hết (Run All). Code của mọi bước nằm trong thư mục `p2lib/` cạnh notebook, mỗi notebook một module; ô
code ở đây chỉ gọi hàm, in kết quả và so với bản kỳ vọng. Cả bốn notebook mất khoảng mười tiếng trên CPU, phần lớn ở notebook 04 (huấn luyện CNN).

### Cách chạy

1. Tải thư mục chứa notebook này (gồm bốn notebook, `p2lib/`, `expected/` và `requirements.txt`).
2. Cài thư viện: `pip install -r requirements.txt` (Python 3.12). Các phiên bản được ghim vì kết quả phụ thuộc vào chúng, kể cả danh sách hậu tố tên miền
   đóng gói trong `tldextract`.
3. Gói dữ liệu ba corpus ngoài được tải tự động từ Zenodo (hoặc giải nén sẵn vào thư mục `url_data/` cạnh notebook); bộ PhishVN được tải từ Mendeley Data.

### Kiểm như thế nào

Mỗi thí nghiệm ghi một bảng kết quả, mỗi dòng một lần chạy (họ mô hình, giao thức, seed…). Notebook so từng dòng với bảng kỳ vọng cùng tên trong `expected/`
theo các cột khoá, bỏ qua cột thời gian chạy (đồng hồ thật, không lặp lại được). Các số đo được so theo dung sai `MET_TOL` = 10⁻⁴, chặt hơn ba chữ số thập phân
mà bài in: các họ cây (random forest, XGBoost, LightGBM) huấn luyện song song trên nhiều luồng, nên thứ tự cộng số thực, và do đó những chữ số cuối, khác nhau
giữa hai lần chạy (thường lệch cỡ 10⁻⁷ tới 10⁻⁶; một dòng test nằm sát ngưỡng 0,5 mà đổi phía thì F1 lệch cỡ 10⁻⁵). Ma trận chuyển giao làm tròn ba chữ số nên
được so đúng từng ô. Với mỗi phép so, notebook in mức lệch lớn nhất thật sự. Các hàm so nằm trong `p2lib/checks.py`:

- `check_table(bước, tên_file, dung_sai)` so một bảng kết quả với bản kỳ vọng cùng tên;
- `check_matrix(bước, tên_file, dung_sai)` so một ma trận chuyển giao từng ô;
- `check_json(bước, kết_quả, file_json, key)` so các con số tính thêm với file JSON bài ghi ra;
- `report(bước)` in số phép khớp của một bước.

### Cấu hình: biến môi trường

Notebook chạy được ngay mà **không cần đặt biến nào**: mọi dữ liệu đều công khai và được tải tự động. Các biến dưới đây chỉ để đổi chỗ lưu
file, ví dụ khi thư mục notebook không có quyền ghi hoặc ổ đĩa không đủ chỗ.

| Biến | Dùng để | Mặc định |
|---|---|---|
| `PHISHVN_WORK_DIR` | thư mục chứa mọi file notebook tải về và ghi ra | `url_work/` cạnh notebook |
| `PHISHVN_DATA_DIR` | thư mục chứa gói dữ liệu Zenodo (tự tải nếu thiếu) | `url_data/` cạnh notebook |
| `PHISHVN_EXPECTED_DIR` | thư mục chứa các file kết quả kỳ vọng để so | `expected/` cạnh notebook, nếu thiếu thì tự tải từ repo công khai |

Có hai cách đặt biến:
1. Đặt trong shell trước khi mở Jupyter, ví dụ `export PHISHVN_WORK_DIR=/duong/dan`.
2. Tạo file `.env` cạnh notebook, mỗi dòng một biến dạng `TEN=gia_tri`. Biến đặt trong shell được ưu tiên hơn biến trong `.env`. Đừng
   commit file `.env`.

Không gõ token trực tiếp vào ô code. Ô thiết lập ở Bước 0 chỉ in biến nào đã được đặt, không in giá trị của token. Các biến này không đổi cách tính: kết
quả giống hệt nhau dù đặt hay không.

## Bước 0: Thiết lập

`setup()` (trong `p2lib/setup.py`) đọc các biến môi trường ở trên, tải bộ PhishVN từ Mendeley Data và ba corpus ngoài từ Zenodo, kiểm mã SHA-256 của từng file,
dựng cây thư mục `url_work/data/processed/` mà mã của bài đọc và ghi, rồi chuyển thư mục làm việc vào đó. Các file đã tải ở notebook trước không tải lại. Mọi kết
quả nằm trong `url_work/data/processed/p2/`, và các notebook sau đọc kết quả của notebook trước từ đó.

In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits

import p2lib
from p2lib import common, core, derived
from p2lib.checks import RES, MET_TOL, CURVE_TOL, THR_TOL, ROUND4_TOL, CNN_TOL, REL_TOL, check, check_table, check_matrix, check_json, report

env = p2lib.setup.setup()

## Bước 1: Dữ liệu và mã dùng chung

**Đặc trưng.** Mỗi URL được mô tả bằng 21 đặc trưng từ vựng của PhishVN (danh sách `COMPPHISH`; tên biến giữ từ trước, lược đồ là của PhishVN): độ dài, số chữ, số chữ số, số ký tự đặc biệt, số dấu chấm,
số tên miền con, độ dài đuôi tên miền, entropy… Trong `vn_compphish.csv` và ba corpus ngoài, đặc trưng đã được tính sẵn.

**Mô hình.** Bảy họ: hồi quy logistic, random forest, HistGB, MLP (`make_model`), cộng XGBoost, LightGBM và CatBoost (`make_any_model`), đều với tham số mặc
định ghi trong mã. Từ bản sửa của bài, mọi họ được huấn luyện **không** cân trọng số lớp (`weighting_params` tắt `class_weight="balanced"` mà ba họ đầu kế
thừa); cấu hình cũ, trộn có và không trọng số, chỉ chạy lại ở Bước 6 để đối chiếu.

**Chỉ số.** F1 tại ngưỡng 0,5; PR-AUC; ROC-AUC; và FPR@R0.90, tỷ lệ báo nhầm khi bắt được 90% URL lừa đảo (`_metrics`). Mỗi lần chạy còn lưu một đường
precision-recall trên lưới recall cố định 200 điểm (`pr_curve_row`); `write_curves` lấy trung bình theo seed.

**Chia theo thời gian.** `load` ghép đặc trưng với ngày phát hiện trong `dataset_url.csv` và tính tên miền đăng ký (`registered_domain`, theo danh sách hậu tố
đóng gói trong `tldextract`, không tải mạng). `split_phishing` lấy 70% URL lừa đảo có ngày sớm nhất làm train, phần còn lại làm test, và bỏ khỏi test mọi URL có
tên miền đăng ký đã xuất hiện trong train (guard). `benign_mask` chia lớp benign ngẫu nhiên theo seed (hoặc theo tên miền nếu `grouped`); `match_prior` lấy mẫu
bớt benign để tỷ lệ lớp của tập test bằng một giá trị cho trước.

**Kiểm định.** So hai cấu hình trên cùng các seed dùng t hiệu chỉnh của Nadeau và Bengio: các lần chia train/test chồng lên nhau nên sai số chuẩn được nới thành
`sqrt((1/K + n_test/n_train) Var(d))`, với `d` là hiệu theo từng seed (`corrected_paired_t`). Khi có nhiều phép so, p được hiệu chỉnh Benjamini-Hochberg
(`bh_adjust`).

Mọi hàm nhắc tới ở trên nằm trong `p2lib/common.py`. Ô dưới nạp corpus một lần cho các bước cần đến nó.

In [ ]:
df_all = common.load()
print(f"{len(df_all):,} URLs, {int(df_all.y.sum()):,} phishing; dated phishing: {int(((df_all.y == 1) & df_all.date.notna()).sum()):,}")

## Bước 2: Benchmark bảy họ

Hai giao thức trên toàn bộ 53.116 URL: `random` chia ngẫu nhiên phân tầng 70/30, lặp 5 seed (cách đánh giá được công bố nhiều nhất); `temporal` dùng cột
`split` có sẵn trong bộ dữ liệu (cách chia chính thức của bộ PhishVN). Hồi quy logistic không phụ thuộc seed nên giao thức `temporal` chỉ chạy một lần.

In [ ]:
core.run_benchmark()
check_table("2 benchmark", "p2_benchmark.csv", MET_TOL)
check_table("2 benchmark", "p2_pr_curves.csv", CURVE_TOL)
report("2 benchmark")

## Bước 3: Chia theo thời gian nghiêm ngặt

Chỉ dùng URL lừa đảo có ngày. Giao thức `temporal_strict`: 70% URL lừa đảo sớm nhất làm train, phần sau làm test, có guard tên miền (Bước 1); lớp benign chia
ngẫu nhiên 70/30 theo seed. Đối chứng `random_same_rows`: cùng các dòng đó nhưng URL lừa đảo được chia ngẫu nhiên, với cùng tỷ lệ. Hiệu giữa hai giao thức là phần
do thời gian gây ra, khi tập dòng được giữ nguyên.

In [ ]:
core.run_strict()
check_table("3 temporal strict", "p2_temporal_strict.csv", MET_TOL)
check_table("3 temporal strict", "p2_pr_curves_strict.csv", CURVE_TOL)
report("3 temporal strict")

## Bước 4: Các đối chứng của giao thức thời gian

Giao thức thời gian có guard tên miền ở phía lừa đảo, còn đối chứng ngẫu nhiên ở Bước 3 thì không, nên một phần chênh lệch giữa hai bên là do guard. Bốn biến
thể đo phần đó:
- `guarded`: đối chứng ngẫu nhiên cũng có guard (`random_same_rows_guarded`), để giao thức là biến duy nhất;
- `guarded_pm`: thêm việc lấy mẫu bớt benign trong test của đối chứng cho tỷ lệ lớp bằng bên thời gian;
- `guarded_pm_bgroup`: như trên, và lớp benign được chia theo tên miền đăng ký (mỗi tên miền nằm trọn một phía);
- `bgroup`: giao thức gốc của Bước 3 với benign chia theo tên miền.

In [ ]:
for out, kw in (("p2_temporal_strict_guarded.csv", dict(guard_control=True)),
                ("p2_temporal_strict_guarded_pm.csv", dict(guard_control=True, match_prior=True)),
                ("p2_temporal_strict_guarded_pm_bgroup.csv", dict(guard_control=True, match_prior=True, benign_grouped=True)),
                ("p2_temporal_strict_bgroup.csv", dict(benign_grouped=True))):
    core.run_strict(out=f"{RES}/{out}", curves="", **kw)
    check_table("4 guard controls", out, MET_TOL)
report("4 guard controls")

## Bước 5: CatBoost với 20 seed

Giao thức của Bước 3, chỉ CatBoost, 20 seed: dùng cho phép so có cặp với stacking ở notebook 03, cần nhiều lần chia hơn năm.

In [ ]:
core.run_strict(families=["CatBoost"], seeds=20, out=f"{RES}/p2_temporal_strict_cb_k20.csv", curves="")
check_table("5 CatBoost k20", "p2_temporal_strict_cb_k20.csv", MET_TOL)
report("5 CatBoost k20")

## Bước 6: Cấu hình cũ có trọng số lớp

Trước bản sửa, hồi quy logistic, random forest và HistGB được huấn luyện với `class_weight="balanced"`, còn bốn họ kia thì không. Bước này chạy lại Bước 2 và 3
theo cấu hình trộn đó, chỉ để bài đối chiếu: riêng việc cân trọng số đã tạo ra cả khoảng cách 5,3 điểm F1 giữa các họ mà bản đầu của bài từng nêu.

`common.legacy_weighting()` bật lại cấu hình cũ trong khối `with` và tắt nó khi ra khỏi khối.

In [ ]:
with common.legacy_weighting():
    core.run_benchmark(out=f"{RES}/p2_benchmark_mixedweights.csv", curves="", unweighted=False)
    core.run_strict(out=f"{RES}/p2_temporal_strict_mixedweights.csv", curves="")
check_table("6 legacy weighting", "p2_benchmark_mixedweights.csv", MET_TOL)
check_table("6 legacy weighting", "p2_temporal_strict_mixedweights.csv", MET_TOL)
report("6 legacy weighting")

## Bước 7: Tỷ lệ lớp hay tập dòng

Khi chuyển từ toàn bộ corpus sang các dòng có ngày, hai thứ đổi cùng lúc: tỷ lệ lừa đảo (0,69 xuống 0,55) và chính tập URL lừa đảo. Bốn arm tách hai thứ đó bằng
lấy mẫu lại, mỗi arm chia ngẫu nhiên phân tầng: toàn bộ ở tỷ lệ gốc; toàn bộ với lớp lừa đảo rút bớt về đúng số dòng có ngày; các dòng có ngày ở tỷ lệ của chúng;
và các dòng có ngày với benign rút bớt về tỷ lệ của toàn bộ. Ngoài F1 tại 0,5, mỗi lần chạy còn ghi F1 lớn nhất trên đường PR (`maxF1`).

Bảng của bài được ghi trước khi mã thêm hai cột đếm `n_phish` và `n_benign`, nên phép so dùng các cột chung của hai bảng.

Bảng của bài cũng được chạy với một luồng BLAS. MLP cộng các phép nhân ma trận theo thứ tự phụ thuộc số luồng, nên chạy đa luồng thì các dòng MLP lệch ở chữ số
thứ ba (tới khoảng 0,004); các họ cây không bị ảnh hưởng. Bước này vì vậy chạy dưới `threadpool_limits(1, "blas")`, chậm hơn nhưng khớp đúng.

In [ ]:
with threadpool_limits(1, "blas"):  # the paper's table was written single-threaded; MLP sums depend on it
    core.run_prior_control()
check_table("7 prior or rows", "p2_prior_control.csv", MET_TOL, shared_columns=True)
report("7 prior or rows")

## Bước 8: Ngưỡng chọn trên validation

F1 tại ngưỡng cố định 0,5 phạt những mô hình có điểm số lệch thang. Bước này tách 10% tập train làm validation (theo thời gian với thiết kế thời gian), chọn
ngưỡng tối đa hoá F1 trên đó, rồi đo F1 trên test tại ngưỡng đó (`F1_val`), cạnh F1 tại 0,5 và F1 lớn nhất có thể trên chính test (`F1_oracle`).

Ngưỡng chọn là điểm cực đại trên một đường cong bậc thang, nên một thay đổi cỡ 10⁻¹⁶ trong xác suất của random forest (do cộng song song) có thể đẩy nó sang
điểm kế bên. Vì vậy hai cột phụ thuộc ngưỡng đó (`tau`, `F1_val`) được so với dung sai rộng hơn, `THR_TOL` = 10⁻³; các cột khác giữ `MET_TOL`.

In [ ]:
core.run_val_threshold()
check_table("8 validation threshold", "p2_val_threshold.csv", MET_TOL, col_tol={"tau": THR_TOL, "F1_val": THR_TOL})
report("8 validation threshold")

## Bước 9: Chia theo thời gian cho cả hai lớp

Bước 3 chỉ chia lớp lừa đảo theo thời gian. Ở đây mọi dòng có ngày, cả benign, được cắt tại cùng một ngày (ngày cắt của lớp lừa đảo), có guard tên miền ở cả hai
lớp; đối chứng chia ngẫu nhiên với cùng tỷ lệ train của từng lớp và cùng guard. Hồi quy logistic ở bước này nhạy với cách thư viện BLAS chia luồng, nên bước chạy
với BLAS một luồng, đúng như kết quả của bài.

In [ ]:
with threadpool_limits(1, "blas"):
    core.run_twosided()
check_table("9 two-sided temporal", "p2_temporal_twosided.csv", MET_TOL)
report("9 two-sided temporal")

## Bước 10: Dòng trùng và rò rỉ

Không cần mô hình. Với chia ngẫu nhiên trên toàn bộ corpus, bước này đếm tỷ lệ dòng test có vector đặc trưng giống hệt một dòng train (`test_twin_rate`) và tỷ lệ
có cùng tên miền đăng ký với một dòng train (`test_regdom_shared_rate`); F1 của một "bộ phân loại tra bảng" chỉ nhớ nhãn đa số của từng vector đặc trưng; F1 của
việc đoán mọi thứ là lừa đảo; và trần F1 mà lược đồ đặc trưng cho phép (khi nhiều URL khác nhãn có cùng vector). Sau đó làm lại các tỷ lệ trùng cho hai thiết kế
trên các dòng có ngày.

In [ ]:
core.run_dup_leakage()
check_table("10 duplicates", "p2_dup_leakage.csv", 0)
check_table("10 duplicates", "p2_dup_leakage_protocols.csv", 0)
report("10 duplicates")

## Bước 11: Độ nhạy theo tầng nhãn

Hai giao thức của Bước 3, CatBoost và hồi quy logistic, 5 seed: F1 chung, và tỷ lệ bắt được URL lừa đảo trong từng tầng nhãn. Tầng nhãn gắn liền với nguồn và
thời gian, nên đây là chẩn đoán, không phải tác động nhân quả của tầng.

In [ ]:
core.tier_sensitivity(df_all)
check_table("11 label tiers", "p2_tier_sensitivity.csv", MET_TOL)
report("11 label tiers")

## Bước 12: Các con số tính từ kết quả

Phần lớn con số của bài là trung bình và độ lệch đọc thẳng từ các bảng đã so ở trên. Một số khác được tính thêm từ các bảng đó hoặc từ chính corpus: kiểm định
có cặp, bootstrap, tương quan, và vài phép đếm lại. `derived.core_numbers` (trong `p2lib/derived.py`) tính lại những con số thuộc các bảng của notebook này và so
từng nhóm với `results/p2_generated.json`, file bài ghi các con số đó ra. Mỗi số được coi là khớp nếu lệch không quá `MET_TOL`
hoặc không quá `REL_TOL` = 10⁻³ tính tương đối: thống kê t chia cho một sai số chuẩn nhỏ, nên chữ số cuối lệch do các họ boosting cộng song song bị phóng
to ở đó (bài không in thống kê t, chỉ in hiệu và p). Bốn notebook cùng phủ hết các nhóm của file đó; notebook 04 kiểm lại điều này.

In [ ]:
R = derived.core_numbers(df_all)
for key in R:
    check_json("12 derived numbers", R[key], "results/p2_generated.json", key=key, tol=MET_TOL, rel_tol=REL_TOL)
report("12 derived numbers")

## Bước 13: Tổng hợp

Bảng dưới gom mọi phép kiểm của notebook này theo bước, và lưu chúng vào `data/processed/p2/notebook_checks/` để notebook 04 cộng cả bốn.

In [ ]:
p2lib.checks.summary("01")